# {a}OS Primitive Infographic Prompt Generator

**312 primitives** across **63 parent constructs** across the 7-layer {a}OS reference model.

Each parent construct gets **one infographic prompt** — a self-contained visual that shows:
- The construct name and which layer it belongs to
- All child primitives with 1-line definitions
- A visual metaphor / icon suggestion
- How primitives relate to each other within the construct

### Usage
1. Run all cells to generate the 63 prompts
2. Copy any prompt into NotebookLM, ChatGPT, Midjourney, or any image-gen tool
3. Or run the batch export cell at the bottom to save all prompts to a folder

### Layer Color Key
| Layer | Name | Color | Hex |
|-------|------|-------|-----|
| L7 | Human Interface | Indigo | #818cf8 |
| L6 | Governance | Rose | #fb7185 |
| L5 | Observability | Amber | #fbbf24 |
| L4 | Orchestration | Emerald | #34d399 |
| L3 | Capabilities | Sky | #38bdf8 |
| L2 | Knowledge & Retrieval | Violet | #a78bfa |
| L1 | Infrastructure | Slate | #94a3b8 |

In [ ]:
import json
from collections import OrderedDict
from pathlib import Path

# Load inventory
with open("_primitives_inventory.json", "r", encoding="utf-8") as f:
    primitives = json.load(f)

# Group by parent construct
constructs = OrderedDict()
for p in primitives:
    key = f"{p['layer']} | {p['parent']}"
    constructs.setdefault(key, []).append(p)

LAYER_NAMES = {
    "L7": "Human Interface",
    "L6": "Governance",
    "L5": "Observability",
    "L4": "Orchestration",
    "L3": "Capabilities",
    "L2": "Knowledge & Retrieval",
    "L1": "Infrastructure",
}

LAYER_COLORS = {
    "L7": "#818cf8",  # Indigo
    "L6": "#fb7185",  # Rose
    "L5": "#fbbf24",  # Amber
    "L4": "#34d399",  # Emerald
    "L3": "#38bdf8",  # Sky
    "L2": "#a78bfa",  # Violet
    "L1": "#94a3b8",  # Slate
}

print(f"Loaded {len(primitives)} primitives in {len(constructs)} parent constructs")
for key, prims in constructs.items():
    print(f"  {key} ({len(prims)} primitives)")

## Infographic Style System Prompt

This is the **shared system prompt** prepended to every infographic generation request. It locks the visual identity.

In [ ]:
SYSTEM_PROMPT = """You are a visual design assistant creating ADHD-friendly technical infographics for the {a}OS Agentic Operating System reference model.

DESIGN RULES:
- Dark background (#0a0a0f) with high-contrast text
- Use the layer accent color as the primary highlight
- Clean, minimal layout — no clutter
- Large readable typography (minimum 14pt equivalent)
- Each primitive shown as a distinct card/node with its name + 1-line description
- Show relationships between primitives with arrows or connectors if applicable
- Include a small icon or visual metaphor for each primitive
- Include the layer badge (e.g. "L4 · Orchestration") in the top-left corner
- Parent construct name is the large title
- Aspect ratio: 16:9 (landscape)
- Style: flat design, subtle gradients, no 3D, modern SaaS dashboard aesthetic
- Footer: "{a}OS Reference Model · vitaminR" in small text

FORMAT: Create a single infographic image. Do NOT return markdown or text descriptions.
"""

print("System prompt loaded (shared across all 63 infographics)")
print(f"Length: {len(SYSTEM_PROMPT)} chars")

## Generate All 63 Infographic Prompts

In [ ]:
def build_prompt(construct_key: str, prims: list[dict]) -> str:
    """Build a single infographic prompt for one parent construct."""
    layer = prims[0]["layer"]
    parent = prims[0]["parent"]
    layer_name = LAYER_NAMES.get(layer, "Unknown")
    color = LAYER_COLORS.get(layer, "#888")

    prim_lines = []
    for p in prims:
        desc = p.get("desc", "No description.")
        prim_lines.append(f"  - **{p['name']}**: {desc}")

    prompt = f"""Create an infographic for the \"{parent}\" construct from the {{a}}OS Agentic Operating System reference model.

LAYER: {layer} — {layer_name}
ACCENT COLOR: {color}
CONSTRUCT: {parent}
PRIMITIVE COUNT: {len(prims)}

PRIMITIVES (show each as a card/node):
{chr(10).join(prim_lines)}

VISUAL GUIDANCE:
- Title: \"{parent}\" in large text, colored {color}
- Layer badge: \"{layer} · {layer_name}\" top-left
- Show all {len(prims)} primitives as connected cards on the dark (#0a0a0f) background
- Use {color} accent for borders, connectors, and highlights
- Each card: primitive name (bold) + 1-line description (smaller)
- If primitives have a natural flow or hierarchy, show it with arrows
- Keep it clean, scannable, ADHD-friendly — no walls of text
"""
    return prompt.strip()

# Generate all prompts
all_prompts = OrderedDict()
for key, prims in constructs.items():
    all_prompts[key] = build_prompt(key, prims)

print(f"Generated {len(all_prompts)} infographic prompts")
print(f"\nFirst prompt preview ({list(all_prompts.keys())[0]}):\n")
print(list(all_prompts.values())[0])

## Browse All Prompts

Run this cell to see all 63 prompts with separators.

In [ ]:
for i, (key, prompt) in enumerate(all_prompts.items(), 1):
    layer = key.split(" | ")[0]
    parent = key.split(" | ")[1]
    count = len(constructs[key])
    print(f"\n{'='*80}")
    print(f"INFOGRAPHIC {i:02d}/{len(all_prompts)} — {key} ({count} primitives)")
    print(f"{'='*80}")
    print(prompt)
    print()

## Export All Prompts

Saves each prompt to a separate `.txt` file for batch processing, plus one combined file.

In [ ]:
import re

out_dir = Path("docs/infographic-prompts")
out_dir.mkdir(parents=True, exist_ok=True)

# Individual files
for i, (key, prompt) in enumerate(all_prompts.items(), 1):
    layer = key.split(" | ")[0].lower()
    parent = key.split(" | ")[1]
    slug = re.sub(r"[^a-z0-9]+", "-", parent.lower()).strip("-")
    filename = f"{i:02d}_{layer}_{slug}.txt"
    filepath = out_dir / filename
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(f"SYSTEM PROMPT:\n{SYSTEM_PROMPT}\n\nUSER PROMPT:\n{prompt}\n")

# Combined file
combined = out_dir / "_ALL_PROMPTS.md"
with open(combined, "w", encoding="utf-8") as f:
    f.write("# {a}OS Infographic Prompts — All 63 Constructs\n\n")
    f.write(f"Generated from 312 primitives across 7 layers.\n\n")
    f.write(f"## System Prompt (shared)\n\n```\n{SYSTEM_PROMPT}\n```\n\n")
    f.write("---\n\n")
    for i, (key, prompt) in enumerate(all_prompts.items(), 1):
        count = len(constructs[key])
        f.write(f"## {i:02d}. {key} ({count} primitives)\n\n")
        f.write(f"```\n{prompt}\n```\n\n---\n\n")

print(f"Exported {len(all_prompts)} individual prompt files to {out_dir}/")
print(f"Combined file: {combined}")
print(f"\nFiles:")
for p in sorted(out_dir.glob("*.txt")):
    print(f"  {p.name}")

## NotebookLM Source Document

Run this cell to generate a single markdown document optimized for uploading to **Google NotebookLM** as a source. It contains every primitive definition structured for NotebookLM's RAG to understand.

In [ ]:
nlm_path = Path("docs/aOS_primitives_for_notebooklm.md")

with open(nlm_path, "w", encoding="utf-8") as f:
    f.write("# {a}OS Agentic Operating System — Complete Primitive Reference\n\n")
    f.write("This document contains all 312 primitives from the {a}OS 7-layer reference model, ")
    f.write("grouped by parent construct. Each primitive is a named, typed building block ")
    f.write("that agentic systems use at a specific layer of the stack.\n\n")
    f.write("## Layer Overview\n\n")
    f.write("| Layer | Name | Description |\n")
    f.write("|-------|------|-------------|\n")
    layer_descs = {
        "L7": "Human-facing interaction: intent parsing, session state, feedback",
        "L6": "Security, policy enforcement, guardrails, compliance, audit",
        "L5": "Logging, metrics, evaluation, cost tracking, drift detection",
        "L4": "Multi-agent coordination, task routing, state management",
        "L3": "Tool use, code execution, APIs, browser, file system, messaging",
        "L2": "Search, embeddings, chunking, reranking, retrieval",
        "L1": "Model serving, tokenization, GPU infra, fine-tuning",
    }
    for layer in ["L7", "L6", "L5", "L4", "L3", "L2", "L1"]:
        f.write(f"| {layer} | {LAYER_NAMES[layer]} | {layer_descs[layer]} |\n")
    f.write("\n---\n\n")

    # Group by layer then parent
    for layer in ["L7", "L6", "L5", "L4", "L3", "L2", "L1"]:
        f.write(f"## {layer} — {LAYER_NAMES[layer]}\n\n")
        layer_constructs = {k: v for k, v in constructs.items() if k.startswith(layer)}
        for key, prims in layer_constructs.items():
            parent = prims[0]["parent"]
            f.write(f"### {parent}\n\n")
            f.write(f"Parent construct in {layer} ({LAYER_NAMES[layer]}). Contains {len(prims)} primitives.\n\n")
            for p in prims:
                desc = p.get("desc", "No description available.")
                f.write(f"- **{p['name']}**: {desc}\n")
            f.write("\n")
        f.write("---\n\n")

print(f"NotebookLM source saved to {nlm_path}")
print(f"Size: {nlm_path.stat().st_size:,} bytes")
print(f"\nUpload this file to NotebookLM as a source, then ask it:")
print('  - "Create an infographic for the Execution Plan construct"')
print('  - "Explain the relationship between L4 primitives"')
print('  - "What primitives does an agent need for tool use?"')

## Summary Stats

In [ ]:
from collections import Counter

layer_counts = Counter(p["layer"] for p in primitives)
construct_counts = Counter(f"{p['layer']} | {p['parent']}" for p in primitives)

print("=" * 60)
print("  {{a}}OS INFOGRAPHIC PROMPT INVENTORY")
print("=" * 60)
print(f"  Total primitives:         {len(primitives)}")
print(f"  Parent constructs:        {len(constructs)}")
print(f"  Infographic prompts:      {len(all_prompts)}")
print(f"  Layers covered:           {len(layer_counts)}")
print()
print("  By layer:")
for layer in ["L7", "L6", "L5", "L4", "L3", "L2", "L1"]:
    lc = layer_counts.get(layer, 0)
    cc = sum(1 for k in constructs if k.startswith(layer))
    print(f"    {layer} {LAYER_NAMES[layer]:.<25} {lc:3d} primitives in {cc:2d} constructs")
print("=" * 60)